In [ ]:
import yfinance as yf
import pandas as pd
from fear_and_greed import FearAndGreedIndex
from datetime import datetime
import os
import time

# ============ CONFIG ============
cache_file_name = os.path.join('data_cache/bitcoin_data.pkl')
start_date = pd.to_datetime('2014-09-17') #First available date: 2014-09-17
end_date = pd.to_datetime('2026-03-01') #Freeze date
coinMetrics_drop_cols = [
    'AssetCompletionTime', 'AssetEODCompletionTime',
    'BlkCnt', 'IssTotNtv', 'IssTotUSD', 'SplyExpFut10yr', # deterministic
    'FlowInExUSD', 'FlowOutExUSD', 'SplyExUSD', # redundant with Native values
    'PriceBTC',                        # always 1
    'PriceUSD','ROI30d', 'volume_reported_spot_usd_1d', # already have it
    'ReferenceRate', 'ReferenceRateUSD', 'ReferenceRateETH', 'ReferenceRateEUR', # empty util 21/06/2019
    'CapMrktEstUSD' # empty until 22/06/2019
]
coinMetrics_archive_url = "https://raw.githubusercontent.com/coinmetrics/data/master/csv/btc.csv"

# Not using BGeometrics, only 8 metrics/hour cap. Time invested not worth the return
# BGeometrics_metrics = [
#     "mvrv-zscore",
#     "nupl",
#     "sopr",
#     "puell-multiple",
#     "reserve-risk",
#     "realised-price",
#     "exchange-netflow",
#     "hashribbons",      
# ]
# ==================================

os.makedirs(cache_dir, exist_ok=True)

# ============ DOWNLOAD ============
# Download Bitcoin price history
bitcoin_price_history = yf.download('BTC-USD', start=start_date, end=end_date, progress=False, keepna = True,  multi_level_index=False)
    
# Download Fear and Greed Index data
fgi = FearAndGreedIndex()
fgi_raw = fgi.get_historical_data(start_date)
fgi_df = pd.DataFrame(fgi_raw)
fgi_df['Date'] = pd.to_datetime(fgi_df['timestamp'].astype(int), unit='s')
# Convert Fear and Greed values to integers (0-100 scale)
fgi_df['value'] = fgi_df['value'].astype(int)

# On-chain data from Coin Metrics
metrics_df = pd.read_csv(coinMetrics_archive_url, index_col='time', parse_dates=True, low_memory=False)
metrics_df.index = pd.to_datetime(metrics_df.index)
metrics_df = metrics_df.drop(columns=coinMetrics_drop_cols)

# # Not using BGeometrics, only 8 metrics/hour cap. Time invested not worth the return
    # TOKEN = " " # log into account and get token

    # BGeometrics_metrics_dict = {}
    # for metric in BGeometrics_metrics:
    #     try:
    #         url = f"https://bitcoin-data.com/v1/{metric}/csv?token={TOKEN}"
    #         BGeometrics_metrics_dict[metric] = pd.read_csv(url)
    #     except Exception as e:
    #         print(f"✗ {metric}: {e}")
    
    
# Fear & Greed Index - market sentiment indicator (0=Extreme Fear, 100=Extreme Greed)
fgi_df = fgi_df.set_index('Date')

# ============ HANDLE DATA GAPS ============
# Add missing dates - not changing raw data start/end dates
full_index_bitcoin_price_history = pd.date_range(start=start_date, end=end_date, freq="D")
bitcoin_price_history = bitcoin_price_history.reindex(full_index_bitcoin_price_history)

full_index_fgi_df = pd.date_range(start=min(fgi_df.index), end=max(fgi_df.index), freq="D")
fgi_df = fgi_df.reindex(full_index_fgi_df)

full_index_metrics_df = pd.date_range(start=min(metrics_df.index), end=max(metrics_df.index), freq="D")
metrics_df = metrics_df.reindex(full_index_metrics_df)

In [ ]:
# ============ SAVE CACHE ============
cache_data = {
    'bitcoin_price_history': bitcoin_price_history,
    'fgi_data': fgi_df,
    'metrics_data': metrics_df,
    'cache_date': datetime.now().strftime('%Y-%m-%d %H:%M')
}
pd.to_pickle(cache_data, cache_file_name)
print(f"    Cached to: {cache_file_name}")
print(f"    Range: {start_date.date()} to {end_date.date()}")